# Likelihood and the score

## What the likelihood function is, why it is not a probability, and what the score tells you

The same formula $f(x, \theta)$ can be read in two entirely different ways. As a function of $x$ for fixed $\theta$, it is a probability model: it tells you how likely different data values are under a given parameter. As a function of $\theta$ for fixed observed $x$, it is the likelihood function: it tells you which parameter values make the observed data more or less plausible.

Most confusion about likelihood comes from mixing up what is random and what is fixed. This notebook explains the distinction carefully, and then follows it to its natural consequences: the log-likelihood, the score function, maximum likelihood estimation, and the population interpretation via KL divergence.

All figures are saved into `content/blog/likelihood-and-score/` next to `index.md` (Hugo page bundle). After regenerating PNGs, run **`hugo`** from the repository root so those files are copied into `public/…/likelihood-and-score/` — otherwise the site can show broken images if it was built before the PNGs existed.

In [1]:
from __future__ import annotations

from math import erf, pi, sqrt
from pathlib import Path

import matplotlib

matplotlib.use("Agg")  # headless backend — avoids blank PNGs on some Windows setups

import matplotlib.pyplot as plt
import numpy as np
from scipy.special import gammaln
from scipy.stats import poisson as poisson_dist

for _style in ("seaborn-v0_8-whitegrid", "seaborn-whitegrid", "ggplot"):
    try:
        plt.style.use(_style)
        break
    except OSError:
        continue
plt.rcParams["figure.dpi"] = 140
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

RNG = np.random.default_rng(20260407)


def locate_root() -> Path:
    cwd = Path.cwd().resolve()
    if (cwd / "content").exists() and (cwd / "notebooks").exists():
        return cwd
    if cwd.name == "notebooks" and (cwd.parent / "content").exists():
        return cwd.parent
    for parent in [cwd, *cwd.parents]:
        if (parent / "content").exists() and (parent / "notebooks").exists():
            return parent
        if (parent / "hugo.toml").exists() and (parent / "content").exists():
            return parent
    raise RuntimeError("Could not locate repository root.")


ROOT = locate_root()
OUTDIR = ROOT / "content" / "blog" / "likelihood-and-score"
OUTDIR.mkdir(parents=True, exist_ok=True)


def normal_pdf(x, mean=0.0, sd=1.0):
    z = (x - mean) / sd
    return np.exp(-0.5 * z**2) / (sd * sqrt(2 * pi))


def poisson_loglik(theta_grid, data):
    # Log-likelihood for i.i.d. Poisson data over a grid of theta values.
    n = len(data)
    sum_x = data.sum()
    const = gammaln(data + 1).sum()
    return -n * theta_grid + sum_x * np.log(theta_grid) - const


def savefig(name):
    path = (OUTDIR / name).resolve()
    path.parent.mkdir(parents=True, exist_ok=True)
    plt.tight_layout()
    plt.savefig(path, bbox_inches="tight", dpi=180, facecolor="white")
    plt.close()
    print(path.relative_to(ROOT))
    return path


ROOT, OUTDIR

(WindowsPath('C:/Users/Alex/OneDrive - University of Cambridge/Desktop/Blog/alex-blog'),
 WindowsPath('C:/Users/Alex/OneDrive - University of Cambridge/Desktop/Blog/alex-blog/content/blog/likelihood-and-score'))

## The likelihood function

Start from a parametric model $\{f(\cdot, \theta) : \theta \in \Theta\}$. For each fixed $\theta$, the function $x \mapsto f(x, \theta)$ is a density or probability mass function on the sample space: it assigns probability (or probability density) to data values.

Now suppose we observe data $x$. Define the **likelihood function** by

$$
L(\theta; x) = f(x, \theta),
$$

regarded as a function of $\theta$ for fixed $x$. The algebraic expression is identical, but the roles of the two arguments have reversed: $x$ is now held constant and $\theta$ is the variable.

> Probability asks: if the parameter were $\theta$, how likely are different data values?
> Likelihood asks: given the data we saw, which parameter values make them more plausible?

The word "likelihood" is about *relative support* among parameter values. It does not assign probabilities to parameter values. Saying "$L(\theta_1; x) > L(\theta_2; x)$" means that the observed data are more consistent with $\theta_1$ than with $\theta_2$ under the model — nothing more.

A common convention is to define the likelihood only up to positive proportionality: multiplying by any positive constant that does not depend on $\theta$ changes nothing about relative comparisons or the location of the maximum.

In [2]:
# Figure 1: same formula, two roles — density in x versus likelihood in mu
mu0 = 2.0
x0 = 1.2
x_grid = np.linspace(-2, 6, 500)
mu_grid = np.linspace(-2, 6, 500)

fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.4))

y_dens = normal_pdf(x_grid, mu0, 1.0)
axes[0].plot(x_grid, y_dens, color="#1d4ed8", lw=2)
axes[0].fill_between(x_grid, y_dens, alpha=0.15, color="#93c5fd")
axes[0].axvline(x0, color="#dc2626", ls="--", lw=1.5, label=rf"$x_0 = {x0}$")
axes[0].set_xlabel(r"$x$")
axes[0].set_ylabel(r"$f(x;\, \mu_0)$")
axes[0].set_title(rf"density in $x$ for fixed $\mu_0 = {mu0:.0f}$", fontsize=11)
axes[0].legend(frameon=False, fontsize=9)

y_lik = normal_pdf(x0, mu_grid, 1.0)
axes[1].plot(mu_grid, y_lik, color="#1d4ed8", lw=2)
axes[1].axvline(x0, color="#dc2626", ls="--", lw=1.5, label=rf"$x_0 = {x0}$")
axes[1].set_xlabel(r"$\mu$")
axes[1].set_ylabel(r"$L(\mu;\, x_0)$")
axes[1].set_title(rf"likelihood in $\mu$ for fixed $x_0 = {x0}$", fontsize=11)
axes[1].legend(frameon=False, fontsize=9)

savefig("density-vs-likelihood.png")

content\blog\likelihood-and-score\density-vs-likelihood.png


WindowsPath('C:/Users/Alex/OneDrive - University of Cambridge/Desktop/Blog/alex-blog/content/blog/likelihood-and-score/density-vs-likelihood.png')

## Likelihood for an i.i.d. sample

If $X_1, \dots, X_n \stackrel{\text{i.i.d.}}{\sim} f(\cdot, \theta)$, then the joint density is the product of the individual densities, and the likelihood becomes

$$
L_n(\theta; x_1, \dots, x_n) = \prod_{i=1}^n f(x_i, \theta).
$$

Independence turns the plausibility of the whole sample into a product of per-observation plausibilities. Taking logarithms gives the **log-likelihood**

$$
\ell_n(\theta) = \log L_n(\theta) = \sum_{i=1}^n \log f(x_i, \theta),
$$

and the **average log-likelihood**

$$
\bar\ell_n(\theta) = \frac{1}{n} \ell_n(\theta) = \frac{1}{n} \sum_{i=1}^n \log f(x_i, \theta).
$$

> The log-likelihood turns a complicated product of many little plausibility contributions into an additive bookkeeping system.

The average log-likelihood $\bar\ell_n$ is especially important: it is a sample average, so by the law of large numbers it should converge to an expectation. This bridge to asymptotics is the key to understanding why maximum likelihood works.

## What is random and what is fixed?

The likelihood depends on the data, and the data are random. This makes the likelihood itself a random object — but the randomness needs to be understood precisely.

### Before observing data

The random variables $X_1, \dots, X_n$ have not yet been realised. The likelihood $L_n(\theta; X_1, \dots, X_n)$ is a random function of $\theta$: for each outcome $\omega$ of the experiment, the realised data $X_1(\omega), \dots, X_n(\omega)$ produce a different curve

$$
\theta \mapsto L_n(\theta; X_1(\omega), \dots, X_n(\omega)).
$$

In this sense, the likelihood is a random element of a function space.

### After observing data

Once the data are observed as $x_1, \dots, x_n$, the likelihood $L_n(\theta; x_1, \dots, x_n)$ is just an ordinary deterministic curve over $\Theta$. There is nothing random about it.

> The likelihood is random only because it depends on random data. Once the data are realised, the likelihood curve itself is fixed.

The analogy with estimators is helpful. Before data, an estimator $\hat\theta(X)$ is a random variable; after data, $\hat\theta(x)$ is a number. Likewise, before data, the likelihood is a random function; after data, it is a concrete curve.

The phrase "random curve" does *not* mean the parameter is random. It means the graph you end up with depends on which sample you happened to draw.

In [3]:
# Figure 2: many normalised log-likelihood curves from repeated Poisson samples
theta0 = 3.0
n_obs = 20
n_samples = 20
theta_grid = np.linspace(0.5, 7.0, 500)

fig, ax = plt.subplots(figsize=(8.3, 4.8))
for _ in range(n_samples):
    data = RNG.poisson(theta0, size=n_obs)
    loglik = poisson_loglik(theta_grid, data)
    ax.plot(theta_grid, loglik - loglik.max(), color="#2563eb", alpha=0.4, lw=1.2)

ax.axvline(theta0, color="#dc2626", ls="--", lw=1.5,
           label=rf"true $\theta_0 = {theta0:.0f}$")
ax.set_xlabel(r"$\theta$")
ax.set_ylabel(r"$\ell_n(\theta) - \max\, \ell_n$")
ax.legend(frameon=False, fontsize=10)
savefig("random-likelihood-curves.png")

content\blog\likelihood-and-score\random-likelihood-curves.png


WindowsPath('C:/Users/Alex/OneDrive - University of Cambridge/Desktop/Blog/alex-blog/content/blog/likelihood-and-score/random-likelihood-curves.png')

## Why likelihood is not a probability distribution in $\theta$

A likelihood function is not a probability density or mass function over $\theta$, for several reasons.

**The parameter is not random in the frequentist setup.** Probability is defined for random quantities on a probability space. In the standard parametric model, the randomness lives in $X$, not in $\theta$. The parameter is an unknown constant.

**The likelihood does not integrate to 1 over $\theta$.** It may integrate to something other than 1, or it may diverge entirely. Consider a single Bernoulli observation $x \in \{0, 1\}$:

$$
L(p; x) = p^x (1-p)^{1-x}, \quad p \in [0,1].
$$

When $x = 1$, $\int_0^1 L(p; 1)\, dp = \int_0^1 p\, dp = 1/2$, not $1$. In other models the integral over parameter space may not even converge.

**Its scale is arbitrary.** The likelihood is defined only up to positive multiplicative constants independent of $\theta$. An object whose numerical scale is arbitrary cannot be a probability density.

**It compares parameter values; it does not assign probabilities to them.** The likelihood tells you that $\theta_1$ is twice as well supported as $\theta_2$, but it does not say that the probability of $\theta_1$ is anything. If you want a probability distribution over $\theta$, you need a prior and a posterior.

The deeper mathematical point: a density is a density with respect to some variable and some measure. For fixed $\theta$, $x \mapsto f(x, \theta)$ is a density with respect to a measure on the sample space $\mathcal{X}$. When you freeze $x$ and let $\theta$ vary, you have changed which space the function lives on. There is no automatic dominating measure on parameter space $\Theta$ with respect to which $L(\theta; x)$ would be a probability density.

## The log-likelihood

The log-likelihood is not merely a computational convenience. It changes the mathematical character of the problem:

1. **Products become sums.** The log-likelihood for an i.i.d. sample is $\ell_n(\theta) = \sum_{i=1}^n \log f(x_i, \theta)$, a sum of per-observation contributions.
2. **Numerical stability.** Products of many small numbers underflow in floating-point arithmetic; sums of their logarithms do not.
3. **Derivatives simplify.** The derivative of a sum is a sum of derivatives, far simpler than the derivative of a product.
4. **The maximiser is unchanged.** Since $\log$ is strictly increasing, $\arg\max_\theta L_n(\theta) = \arg\max_\theta \ell_n(\theta)$.

> Likelihood is multiplicative, but statistical thinking is usually additive.

In [4]:
# Figure 3: likelihood versus log-likelihood for one Poisson sample
theta0_fig3 = 3.0
n_fig3 = 20
data_fig3 = RNG.poisson(theta0_fig3, size=n_fig3)
theta_grid_fig3 = np.linspace(0.3, 8, 500)

loglik_fig3 = poisson_loglik(theta_grid_fig3, data_fig3)
lik_fig3 = np.exp(loglik_fig3 - loglik_fig3.max())
mle_fig3 = data_fig3.mean()

fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.4))

axes[0].plot(theta_grid_fig3, lik_fig3, color="#1d4ed8", lw=2)
axes[0].axvline(mle_fig3, color="#dc2626", ls="--", lw=1.5,
                label=rf"MLE $= {mle_fig3:.1f}$")
axes[0].set_xlabel(r"$\theta$")
axes[0].set_ylabel(r"$L_n(\theta)$ (normalised)")
axes[0].set_title("likelihood", fontsize=11)
axes[0].legend(frameon=False, fontsize=9)

axes[1].plot(theta_grid_fig3, loglik_fig3, color="#1d4ed8", lw=2)
axes[1].axvline(mle_fig3, color="#dc2626", ls="--", lw=1.5,
                label=rf"MLE $= {mle_fig3:.1f}$")
axes[1].set_xlabel(r"$\theta$")
axes[1].set_ylabel(r"$\ell_n(\theta)$")
axes[1].set_title("log-likelihood", fontsize=11)
axes[1].legend(frameon=False, fontsize=9)

savefig("likelihood-vs-loglikelihood.png")

content\blog\likelihood-and-score\likelihood-vs-loglikelihood.png


WindowsPath('C:/Users/Alex/OneDrive - University of Cambridge/Desktop/Blog/alex-blog/content/blog/likelihood-and-score/likelihood-vs-loglikelihood.png')

## Expected log-likelihood and KL divergence

The average log-likelihood $\bar\ell_n(\theta) = \frac{1}{n} \sum_{i=1}^n \log f(X_i, \theta)$ is a sample average. By the law of large numbers, it converges to the **expected log-likelihood** under the true model:

$$
\ell(\theta) = \mathbb{E}_{\theta_0}[\log f(X, \theta)].
$$

This is a deterministic function of $\theta$ that we never observe directly: it is the population target that the empirical curve approximates.

> We maximise the empirical log-likelihood because it is a sample approximation to a population criterion.

If the model is well-specified and identifiable, the expected log-likelihood is maximised at $\theta_0$. The reason is the KL divergence. Write

$$
\ell(\theta) - \ell(\theta_0) = \mathbb{E}_{\theta_0}\!\left[\log \frac{f(X, \theta)}{f(X, \theta_0)}\right] = -\text{KL}(P_{\theta_0} \| P_\theta),
$$

where $\text{KL}(P_{\theta_0} \| P_\theta) = \mathbb{E}_{\theta_0}\!\left[\log \frac{f(X, \theta_0)}{f(X, \theta)}\right] \geq 0$, with equality if and only if $P_\theta = P_{\theta_0}$. Therefore:

- $\ell(\theta) \leq \ell(\theta_0)$ for all $\theta$, with equality exactly when $P_\theta = P_{\theta_0}$.
- Maximising the expected log-likelihood is equivalent to minimising the KL divergence from the true model.

> Expected log-likelihood is population geometry; empirical log-likelihood is its noisy sample version.

Note that KL divergence is not symmetric: $\text{KL}(P \| Q) \neq \text{KL}(Q \| P)$ in general. It is not a metric, but it is always non-negative, and zero only when the two distributions coincide.

In [5]:
# Figure 4: empirical average log-likelihood vs expected log-likelihood
theta0_fig4 = 3.0
n_fig4 = 50
data_fig4 = RNG.poisson(theta0_fig4, size=n_fig4)
theta_grid_fig4 = np.linspace(0.5, 7, 500)

avg_loglik = poisson_loglik(theta_grid_fig4, data_fig4) / n_fig4

x_vals = np.arange(0, 30)
probs = poisson_dist.pmf(x_vals, theta0_fig4)
expected_log_factorial = np.sum(gammaln(x_vals + 1) * probs)
expected_loglik = -theta_grid_fig4 + theta0_fig4 * np.log(theta_grid_fig4) - expected_log_factorial

fig, ax = plt.subplots(figsize=(8.3, 4.8))
ax.plot(theta_grid_fig4, avg_loglik, color="#2563eb", lw=1.8,
        label=r"$\bar\ell_n(\theta)$ (sample, $n=50$)")
ax.plot(theta_grid_fig4, expected_loglik, color="#111827", lw=2, ls="--",
        label=r"$\ell(\theta)$ (population)")
ax.axvline(theta0_fig4, color="#dc2626", ls=":", lw=1.5,
           label=rf"true $\theta_0 = {theta0_fig4:.0f}$")
ax.set_xlabel(r"$\theta$")
ax.set_ylabel("average log-likelihood")
ax.legend(frameon=False, fontsize=10)
savefig("empirical-vs-expected.png")

content\blog\likelihood-and-score\empirical-vs-expected.png


WindowsPath('C:/Users/Alex/OneDrive - University of Cambridge/Desktop/Blog/alex-blog/content/blog/likelihood-and-score/empirical-vs-expected.png')

## Maximum likelihood estimation

The **maximum likelihood estimator** (MLE) is defined as any value $\hat\theta$ satisfying

$$
\hat\theta \in \arg\max_{\theta \in \Theta} L_n(\theta) = \arg\max_{\theta \in \Theta} \ell_n(\theta).
$$

The MLE is the parameter value under which the observed sample is most plausible according to the model. A few important points:

- The MLE need not be unique.
- It may lie on the boundary of $\Theta$.
- Existence and uniqueness are model-dependent.
- Solving $S_n(\theta) = 0$ is often a route to the MLE, but it is not the definition itself.

### Worked example: Poisson

Suppose $X_1, \dots, X_n \stackrel{\text{i.i.d.}}{\sim} \text{Poisson}(\theta)$, $\theta > 0$. The likelihood is

$$
L_n(\theta) = \prod_{i=1}^n \frac{e^{-\theta} \theta^{x_i}}{x_i!} \propto e^{-n\theta} \theta^{\sum x_i}.
$$

The log-likelihood, up to terms not depending on $\theta$, is

$$
\ell_n(\theta) = -n\theta + \left(\sum_{i=1}^n x_i\right) \log\theta + C.
$$

Differentiating:

$$
\ell_n'(\theta) = -n + \frac{\sum_{i=1}^n x_i}{\theta}.
$$

Setting $\ell_n'(\theta) = 0$ gives $\hat\theta = \bar X$. The second derivative is

$$
\ell_n''(\theta) = -\frac{\sum_{i=1}^n x_i}{\theta^2} < 0
$$

whenever the sample is not all zeros, confirming that $\hat\theta = \bar X$ is a maximiser.

## The score function

The **score function** is the derivative of the log-likelihood with respect to the parameter:

$$
S_n(\theta) = \frac{d}{d\theta} \ell_n(\theta).
$$

In higher dimensions, the score is the gradient $S_n(\theta) = \nabla_\theta \ell_n(\theta)$.

The score tells you which way the log-likelihood is sloping at a given parameter value: positive score means the log-likelihood is locally increasing, negative means it is locally decreasing, and zero means you are at a stationary point.

> The likelihood tells you height; the score tells you local direction.

Derivatives are taken with respect to the parameter $\theta$, not the data. The data are fixed once observed; it is $\theta$ that varies.

For the Poisson model, the score for a single observation is

$$
S(\theta) = \frac{d}{d\theta} \log f(X, \theta) = \frac{d}{d\theta}\bigl(-\theta + X\log\theta\bigr) = -1 + \frac{X}{\theta},
$$

and for the full sample:

$$
S_n(\theta) = -n + \frac{\sum_{i=1}^n X_i}{\theta}.
$$

Setting $S_n(\theta) = 0$ recovers the MLE $\hat\theta = \bar X$.

In [6]:
# Figure 5: log-likelihood with score function below, zero crossing marked
theta0_fig5 = 3.0
n_fig5 = 20
data_fig5 = RNG.poisson(theta0_fig5, size=n_fig5)
theta_grid_fig5 = np.linspace(0.5, 7, 500)
mle_fig5 = data_fig5.mean()

loglik_fig5 = poisson_loglik(theta_grid_fig5, data_fig5)
score_fig5 = -n_fig5 + data_fig5.sum() / theta_grid_fig5

fig, axes = plt.subplots(2, 1, figsize=(8.3, 7), sharex=True,
                         gridspec_kw={"height_ratios": [2, 1.5]})

axes[0].plot(theta_grid_fig5, loglik_fig5, color="#1d4ed8", lw=2)
axes[0].axvline(mle_fig5, color="#dc2626", ls="--", lw=1.5,
                label=rf"MLE $= {mle_fig5:.2f}$")
axes[0].set_ylabel(r"$\ell_n(\theta)$")
axes[0].legend(frameon=False, fontsize=10)

axes[1].plot(theta_grid_fig5, score_fig5, color="#1d4ed8", lw=2)
axes[1].axhline(0, color="black", ls="-", lw=0.8)
axes[1].axvline(mle_fig5, color="#dc2626", ls="--", lw=1.5)
axes[1].scatter([mle_fig5], [0], color="#dc2626", zorder=3, s=50)
axes[1].set_xlabel(r"$\theta$")
axes[1].set_ylabel(r"$S_n(\theta)$")

savefig("score-zero-crossing.png")

content\blog\likelihood-and-score\score-zero-crossing.png


WindowsPath('C:/Users/Alex/OneDrive - University of Cambridge/Desktop/Blog/alex-blog/content/blog/likelihood-and-score/score-zero-crossing.png')

## Why the expected score is zero

Under regularity conditions that allow differentiation under the integral sign, the expected score at the true parameter is zero:

$$
\mathbb{E}_\theta[S(\theta)] = \int \frac{\partial}{\partial\theta} \log f(x, \theta) \cdot f(x, \theta)\, dx = \int \frac{\partial}{\partial\theta} f(x, \theta)\, dx = \frac{\partial}{\partial\theta} \int f(x, \theta)\, dx = \frac{\partial}{\partial\theta}(1) = 0.
$$

The key step is the interchange of differentiation and integration, justified by regularity conditions, together with the fact that the density integrates to $1$ for every $\theta$.

> If you stand at the true parameter, the score fluctuates randomly around zero rather than persistently pushing left or right.

This is an important structural property: at the true parameter, the log-likelihood has no systematic first-order tilt. Different samples produce scores that are sometimes positive and sometimes negative at $\theta_0$, but on average they balance out.

For the Poisson model, this is easy to verify directly. The score for one observation is $S(\theta) = -1 + X/\theta$. Since $\mathbb{E}_\theta(X) = \theta$:

$$
\mathbb{E}_\theta[S(\theta)] = -1 + \frac{\mathbb{E}_\theta(X)}{\theta} = -1 + \frac{\theta}{\theta} = 0.
$$

## Sources

1. Rajen Shah, *Principles of Statistics*, University of Cambridge lecture notes, Section 1.1.
2. George Casella and Roger L. Berger, *Statistical Inference*, 2nd edition, Sections 6.3 and 7.2.
3. ["How to rigorously define the likelihood"](https://stats.stackexchange.com/questions/29682/how-to-rigorously-define-the-likelihood), Cross Validated.
4. ["Why isn't the likelihood function a probability density function?"](https://mathoverflow.net/questions/10971/why-isnt-likelihood-a-probability-density-function), MathOverflow.
5. ["Some Principles of Statistical Inference"](https://people.bath.ac.uk/masss/APTS/2022-23/IntroductoryNotes/some-principles-of-statistical-inference.html), University of Bath APTS notes.